# HomeostaticDysregulation

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
4. [Load features](#Load-features)
5. [Load weights into base model](#Load-weights-into-base-model)
6. [Load reference values](#Load-reference-values)
7. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
8. [Check all clock parameters](#Check-all-clock-parameters)
9. [Normal feature ranges](#Normal-feature-ranges)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import subprocess
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.HomeostaticDysregulation)

class HomeostaticDysregulation(pyagingModel):
    """Mahalanobis distance from a young, healthy NHANES III reference cohort."""

    def __init__(self):
        super().__init__()
        for sex in ["male", "female"]:
            for name in ["reference_mean", "reference_sd", "center", "precision", "log_hd_sd"]:
                self.register_buffer(f"{name}_{sex}", torch.empty(0))

    def preprocess(self, x):
        """Apply BioAge's log1p transform to C-reactive protein."""
        return log1p_crp(self.features, x)

    def postprocess(self, x):
        """Score the log Mahalanobis distance from the sex's reference cohort.

        Notes
        -----
        ``center`` is not zero. ``hd_calc`` takes each column's mean and standard
        deviation with ``na.rm = TRUE`` over the full reference column and drops
        incomplete rows only afterwards, so the surviving rows' mean is not the
        centring constant; the residual offset reaches 0.19 standard deviations
        and 

In [3]:
model = pya.models.HomeostaticDysregulation()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = 'homeostaticdysregulation'
model.metadata["data_type"] = 'clinical biomarkers'  # Paper: blood chemistry and organ function test data
model.metadata["species"] = 'Homo sapiens'  # Paper: Homo sapiens
model.metadata["year"] = 2021
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = 'Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for quantification of biological age from blood chemistry and organ function test data: BioAge." GeroScience 43.6 (2021): 2795-2808.'
model.metadata["doi"] = 'https://doi.org/10.1007/s11357-021-00480-5'
model.metadata["notes"] = "Mahalanobis distance from a young, healthy NHANES III reference cohort, fit separately by sex. The output is a log dysregulation score and is NOT expressed in years; larger values mean greater dysregulation. BioAge divides the score by the projection cohort's standard deviation of log(distance), so that NHANES IV constant is baked in per sex to keep single-sample predictions reproducible. C-reactive protein is supplied raw in mg/dL and log1p-transformed inside the clock. Sex is coded female = 1 and male = 0; a dataset with no female column scores every sample with the male reference. An incomplete panel biases the score downward: the score is a distance, so filling an absent biomarker with its reference value removes that marker's own contribution, which lowered the score for every one of the nine markers on average across the reference subjects and by as much as 2.9 on a 1.98-6.76 range. An incomplete panel therefore reads as healthier than it is, so the missing-feature warning the prediction pipeline emits should be heeded."
model.metadata["research_only"] = None
model.metadata["tissue"] = ['blood']  # Paper: blood chemistry
model.metadata["predicts"] = ['biological age']  # Paper: implements three published methods to quantify biological aging based on analysis of chronological age and mortality risk: Klemera-Doubal biological age, PhenoAge, and homeostatic dysregulation
model.metadata["training_target"] = ['not applicable']  # Paper: means = colMeans(ref); cv_mat = var(ref)
model.metadata["unit"] = ['unitless']  # Paper: dat$hd_log = log(hd)/sd(log(hd))
model.metadata["model_type"] = 'Mahalanobis distance composite'  # Paper: hd[x] <- sqrt((dat[x, ] - means) %*% solve(cv_mat) %*% (dat[x, ] - means))
model.metadata["platform"] = ['clinical laboratory assays']  # Paper: blood chemistry and organ function test data
model.metadata["population"] = 'adults'  # Paper: train = NHANES3 %>% filter(age >= 20 & age <= 30 & pregnant == 0 & bmi < 30)
model.metadata["journal"] = 'GeroScience'
model.metadata["last_author"] = 'Daniel W. Belsky'
model.metadata["n_features"] = 10
model.metadata["citations"] = 332
model.metadata["citations_date"] = '2026-08-20'

## Download clock dependencies

In [5]:
# BioAge carries both the fitting functions and the NHANES III / NHANES IV tables they run
# on, so the parameters and the parity reference below are re-derived here rather than read
# from a checked-in copy. The script needs R on PATH; it installs what it is missing into a
# notebook-local library that the Clear directory step removes.
EXTRACT_R = r"""#!/usr/bin/env Rscript
# Derive the homeostatic dysregulation parameters from dayoonkwon/BioAge, which
# ships both the fitting functions and the NHANES III / NHANES IV tables they
# run on.
#
# The fit is trained on SI-unit variants of the NHANES columns, so the
# parameters land natively in pyaging's unit convention. The score is a
# Mahalanobis distance against a z-scored reference, so it is scale invariant
# and fitting in SI units reproduces BioAge's published numbers exactly.
#
# Unit notes, established empirically against the shipped NHANES data:
#   * lncrp is log1p(crp in mg/dL), NOT log(crp): exp(lncrp) - crp == 1 exactly
#     across both cohorts.
#   * albumin_gL == albumin * 10, glucose_mmol == glucose * 0.0555 and
#     creat_umol == creat * 88.4017 are exact, with zero deviation across every
#     non-missing row of both cohorts.
#
# CRP naming. The fitted column stays `log_crp`, because that is what the value
# is: log1p(CRP in mg/dL). The name EMITTED is `c_reactive_protein`, because
# that is what a pyaging user supplies -- the raw measurement in mg/dL, which
# the clock log1p's itself in preprocess(). The rename is name-only: no mean, sd
# or covariance entry moves. The emitted reference rows carry raw `crp`, so
# feeding them in and letting the clock transform reproduces BioAge's output.

local_library <- file.path(getwd(), "Rlib")
dir.create(local_library, showWarnings = FALSE)
.libPaths(c(local_library, .libPaths()))
for (package in c("remotes", "dplyr", "jsonlite")) {
  if (!requireNamespace(package, quietly = TRUE)) {
    install.packages(package, repos = "https://cloud.r-project.org", lib = local_library)
  }
}
if (!requireNamespace("BioAge", quietly = TRUE)) {
  remotes::install_github("dayoonkwon/BioAge@b1f9fc02f086cd4aa74185f2335ab1366082e7fe", lib = local_library, upgrade = "never")
}

suppressPackageStartupMessages({
  library(BioAge)
  library(dplyr)
  library(jsonlite)
})

ALBUMIN_GDL_TO_GL <- 10
CREAT_MGDL_TO_UMOL <- 88.4 # documentation window only; the exact factor is 88.4017
GLUCOSE_MGDL_TO_MMOL <- 0.0555 # 1 / 18.0182

to_pyaging_units <- function(data) {
  data %>% mutate(
    albumin = albumin_gL,
    creatinine = creat_umol,
    glucose = glucose_mmol,
    log_crp = lncrp,
    c_reactive_protein = crp,
    lymphocyte_percent = lymph,
    mean_cell_volume = mcv,
    red_cell_distribution_width = rdw,
    alkaline_phosphatase = alp,
    white_blood_cell_count = wbc,
    female = as.numeric(gender == 2)
  )
}

nhanes4 <- to_pyaging_units(NHANES4)

to_feature_names <- function(names) replace(names, names == "log_crp", "c_reactive_protein")

markers <- c(
  "albumin", "lymphocyte_percent", "mean_cell_volume", "glucose",
  "red_cell_distribution_width", "creatinine", "log_crp",
  "alkaline_phosphatase", "white_blood_cell_count"
)

# Clinically acceptable reference windows. The thresholds are applied to the
# CONVENTIONAL-unit columns exactly as BioAge::hd_nhanes() does, and the NA mask
# is propagated to the SI column, so the reference cohort is bit-identical to
# BioAge's. Applying rounded SI thresholds instead would silently shift cohort
# membership at the boundary (e.g. creat 0.6 mg/dL -> 53.04, not 53, umol/L).
reference_all <- NHANES3 %>%
  filter(age >= 20, age <= 30, pregnant == 0, bmi < 30) %>%
  mutate(
    albumin_gL = ifelse(albumin >= 3.5 & albumin <= 5, albumin_gL, NA),
    alp = ifelse(gender == 2,
      ifelse(alp >= 37 & alp <= 98, alp, NA),
      ifelse(alp >= 45 & alp <= 115, alp, NA)
    ),
    creat_umol = ifelse(gender == 2,
      ifelse(creat >= 0.6 & creat <= 1.1, creat_umol, NA),
      ifelse(creat >= 0.8 & creat <= 1.3, creat_umol, NA)
    ),
    glucose_mmol = ifelse(glucose >= 60 & glucose <= 100, glucose_mmol, NA),
    mcv = ifelse(gender == 2,
      ifelse(mcv >= 78 & mcv <= 101, mcv, NA),
      ifelse(mcv >= 82 & mcv <= 102, mcv, NA)
    ),
    rdw = ifelse(rdw >= 11.5 & rdw <= 14.5, rdw, NA),
    # BioAge filters raw crp < 2 mg/dL; because lncrp == log1p(crp) this is
    # log_crp < log(3), NOT exp(log_crp) < 2.
    lncrp = ifelse(crp < 2, lncrp, NA),
    lymph = ifelse(lymph >= 20 & lymph <= 40, lymph, NA),
    wbc = ifelse(wbc >= 4.5 & wbc <= 11, wbc, NA)
  ) %>%
  to_pyaging_units()

# Documentation only: the same windows expressed in pyaging (SI) units.
reference_window <- function(female_value) {
  list(
    albumin = c(3.5, 5) * ALBUMIN_GDL_TO_GL,
    lymphocyte_percent = c(20, 40),
    mean_cell_volume = if (female_value == 1) c(78, 101) else c(82, 102),
    glucose = c(60, 100) * GLUCOSE_MGDL_TO_MMOL,
    red_cell_distribution_width = c(11.5, 14.5),
    creatinine = if (female_value == 1) {
      c(0.6, 1.1) * CREAT_MGDL_TO_UMOL
    } else {
      c(0.8, 1.3) * CREAT_MGDL_TO_UMOL
    },
    # Expressed in raw mg/dL like the user-facing feature, not on the log1p
    # scale the fit uses: log1p(0) = 0 and log1p(2) = log(3), so this is the
    # exact preimage of the old c(0, log(3)).
    c_reactive_protein = c(0, 2),
    alkaline_phosphatase = if (female_value == 1) c(37, 98) else c(45, 115),
    white_blood_cell_count = c(4.5, 11)
  )
}

# Faithful re-implementation of hd_calc()'s math. Needed because hd_calc returns
# only the cohort-normalized hd/hd_log, never the raw Mahalanobis distance, and
# the normalizing constant sd(log(distance)) has to be baked into the exported
# parameters for single-sample prediction to be reproducible.
fit_for <- function(female_value) {
  reference <- reference_all %>% filter(female == female_value)
  ref_matrix <- as.matrix(reference[, markers])
  ref_mean <- colMeans(ref_matrix, na.rm = TRUE)
  ref_sd <- apply(ref_matrix, 2, sd, na.rm = TRUE)

  standardize <- function(matrix_in) {
    sweep(sweep(matrix_in, 2, ref_mean, "-"), 2, ref_sd, "/")
  }
  ref_z <- na.omit(standardize(ref_matrix))
  center <- colMeans(ref_z)
  covariance <- var(ref_z)

  projection <- na.omit(standardize(as.matrix(
    (nhanes4 %>% filter(female == female_value))[, markers]
  )))
  distances <- sqrt(mahalanobis(projection, center, covariance))

  list(
    biomarkers = to_feature_names(markers),
    reference_mean = unname(ref_mean),
    reference_sd = unname(ref_sd),
    reference_window = reference_window(female_value),
    reference_n = nrow(ref_z),
    standardized_center = unname(center),
    standardized_covariance = unname(as.matrix(covariance)),
    log_hd_sd = sd(log(distances), na.rm = TRUE)
  )
}

fit <- list("0" = fit_for(0), "1" = fit_for(1))

# Sanity check: the re-derived standardized distances must reproduce BioAge's
# own hd_log for the same cohort, given log_hd_sd as the normalizing constant.
for (female_value in c(0, 1)) {
  this_fit <- fit[[as.character(female_value)]]
  cohort <- nhanes4 %>% filter(female == female_value)
  z <- sweep(sweep(as.matrix(cohort[, markers]), 2, this_fit$reference_mean, "-"),
    2, this_fit$reference_sd, "/")
  keep <- stats::complete.cases(z)
  own <- log(sqrt(mahalanobis(
    z[keep, ], this_fit$standardized_center, this_fit$standardized_covariance
  ))) / this_fit$log_hd_sd
  theirs <- hd_calc(
    data = cohort,
    reference = reference_all %>% filter(female == female_value),
    biomarkers = markers
  )$data$hd_log
  theirs <- theirs[!is.na(theirs)]
  cat(sprintf(
    "hd_log parity (female=%d): n=%d max|diff|=%.3g\n",
    female_value, length(own), max(abs(own - theirs))
  ))
  stopifnot(max(abs(own - theirs)) < 1e-10)
}

# ---- Reference predictions, for the parity check ---------------------------
# 20 fixed NHANES IV subjects: complete cases across every column this clock
# consumes, sorted by sampleID (C-locale byte order), first 20. `expected`
# comes from BioAge's own hd_calc, never from a re-implementation.
projected <- bind_rows(lapply(c(0, 1), function(female_value) {
  hd_calc(
    data = nhanes4 %>% filter(female == female_value),
    reference = reference_all %>% filter(female == female_value),
    biomarkers = markers
  )$data %>% select(sampleID, hd_log)
}))

# Subject selection runs over the fitted columns, so carrying the raw CRP column
# cannot shift cohort membership and move `expected`. It is joined back after.
selected <- nhanes4 %>%
  select(sampleID, all_of(c(markers, "female"))) %>%
  filter(stats::complete.cases(.)) %>%
  arrange(sampleID) %>%
  head(20) %>%
  left_join(nhanes4 %>% select(sampleID, c_reactive_protein), by = "sampleID") %>%
  left_join(projected, by = "sampleID")

stopifnot(nrow(selected) == 20, !anyNA(selected))

# The emitted rows carry raw CRP where the fit carries log1p(CRP); the clock
# closes that gap in preprocess(). If this fails, the two have drifted apart.
stopifnot(max(abs(log1p(selected$c_reactive_protein) - selected$log_crp)) < 1e-12)

emit_features <- to_feature_names(c(markers, "female"))

write_json(
  list(
    features = emit_features,
    male = fit[["0"]],
    female = fit[["1"]],
    reference = list(
      sample_ids = selected$sampleID,
      rows = selected %>% select(all_of(emit_features)),
      expected = selected$hd_log
    )
  ),
  "homeostaticdysregulation.json",
  digits = 12, auto_unbox = TRUE, pretty = TRUE
)

cat("wrote homeostaticdysregulation.json\n")
"""

with open("extract_homeostaticdysregulation.R", "w") as handle:
    handle.write(EXTRACT_R)

subprocess.run(["Rscript", "extract_homeostaticdysregulation.R"], check=True)

hd_log parity (female=0): n=18098 max|diff|=8.88e-16


hd_log parity (female=1): n=19485 max|diff|=1.78e-15


wrote homeostaticdysregulation.json


CompletedProcess(args=['Rscript', 'extract_homeostaticdysregulation.R'], returncode=0)

In [6]:
with open("homeostaticdysregulation.json") as handle:
    params = json.load(handle)

params["features"]

['albumin',
 'lymphocyte_percent',
 'mean_cell_volume',
 'glucose',
 'red_cell_distribution_width',
 'creatinine',
 'c_reactive_protein',
 'alkaline_phosphatase',
 'white_blood_cell_count',
 'female']

## Load features

In [7]:
model.features = params["features"]
model.features

['albumin',
 'lymphocyte_percent',
 'mean_cell_volume',
 'glucose',
 'red_cell_distribution_width',
 'creatinine',
 'c_reactive_protein',
 'alkaline_phosphatase',
 'white_blood_cell_count',
 'female']

## Load weights into base model

In [8]:
model.base_model = torch.nn.Identity()

for sex in ["male", "female"]:
    fit = params[sex]
    order = [fit["biomarkers"].index(name) for name in model.features[:-1]]
    for key, buffer in [
        ("reference_mean", "reference_mean"),
        ("reference_sd", "reference_sd"),
        ("standardized_center", "center"),
    ]:
        setattr(model, f"{buffer}_{sex}", torch.tensor([fit[key][index] for index in order], dtype=torch.float64))
    covariance = torch.tensor(fit["standardized_covariance"], dtype=torch.float64)[order][:, order]
    setattr(model, f"precision_{sex}", torch.linalg.inv(covariance))
    setattr(model, f"log_hd_sd_{sex}", torch.tensor(fit["log_hd_sd"], dtype=torch.float64))

    # The reindex above is a no-op whenever the two orders already agree, which is exactly
    # when a mangled copy of it would go unnoticed. Check it mapped what it claims.
    for position, name in enumerate(model.features[:-1]):
        source = fit["biomarkers"].index(name)
        for key, buffer in [
            ("reference_mean", "reference_mean"),
            ("reference_sd", "reference_sd"),
            ("standardized_center", "center"),
        ]:
            assert getattr(model, f"{buffer}_{sex}")[position].item() == fit[key][source], (sex, key, name)

model.center_male

tensor([ 0.1358,  0.0026, -0.0385,  0.0287, -0.1860, -0.0952, -0.0938, -0.0040,
        -0.0027], dtype=torch.float64)

## Load reference values

In [9]:
crp = model.features.index("c_reactive_protein")
centre = {
    sex: getattr(model, f"reference_mean_{sex}") + getattr(model, f"center_{sex}") * getattr(model, f"reference_sd_{sex}")
    for sex in ["male", "female"]
}
sd = {sex: getattr(model, f"reference_sd_{sex}") for sex in ["male", "female"]}
reference = (
    (centre["male"] * sd["female"] + centre["female"] * sd["male"]) / (sd["male"] + sd["female"])
).tolist()
reference[crp] = math.expm1(reference[crp])  # stored raw; preprocess applies log1p

model.reference_values = reference + [0.0]  # female: no sex column means the male reference

assert len(model.reference_values) == len(model.features)

# The residual the compromise leaves, in reference standard deviations.
standardized = torch.tensor(reference, dtype=torch.float64)
standardized[crp] = math.log1p(standardized[crp])
pd.DataFrame(
    {
        "feature": model.features[:-1],
        **{sex: ((standardized - centre[sex]) / sd[sex]).tolist() for sex in ["male", "female"]},
    }
)

,feature,male,female
0,albumin,-0.490255,0.490255
1,lymphocyte_percent,0.011576,-0.011576
2,mean_cell_volume,-0.061514,0.061514
3,glucose,-0.260534,0.260534
4,red_cell_distribution_width,0.097374,-0.097374
5,creatinine,-1.222243,1.222243
6,c_reactive_protein,0.160685,-0.160685
7,alkaline_phosphatase,-0.513524,0.513524
8,white_blood_cell_count,0.057104,-0.057104


## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "log1p_crp"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = "log_mahalanobis_distance"
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for '
             'quantification of biological age from blood chemistry and organ '
             'function test data: BioAge." GeroScience 43.6 (2021): 2795-2808.',
 'citations': 332,
 'citations_date': '2026-08-20',
 'clock_name': 'homeostaticdysregulation',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.1007/s11357-021-00480-5',
 'journal': 'GeroScience',
 'last_author': 'Daniel W. Belsky',
 'model_type': 'Mahalanobis distance composite',
 'n_features': 10,
 'notes': 'Mahalanobis distance from a young, healthy NHANES III reference '
          'cohort, fit separately by sex. The output is a log dysregulation '
          'score and is NOT expressed in years; larger values mean greater '
          "dysregulation. BioAge divides the score by the pr

## Normal feature ranges

In [13]:
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges)

,feature,unit,low,high
0,albumin,g/L,10.00,70.0
1,lymphocyte_percent,%,0.00,100.0
2,mean_cell_volume,fL,40.00,150.0
3,glucose,mmol/L,1.00,60.0
4,red_cell_distribution_width,%,8.00,40.0
5,creatinine,umol/L,10.00,3000.0
6,c_reactive_protein,mg/dL,0.01,50.0
7,alkaline_phosphatase,U/L,5.00,5000.0
8,white_blood_cell_count,10^3 cells/uL,0.05,500.0
9,female,"indicator (1 = female, 0 = male)",0.00,1.0


## Basic test

In [14]:
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = torch.tensor(
    [[(record["low"] + record["high"]) / 2 for record in records]], dtype=torch.float64
)
model.eval()
model.to(torch.float64)
pred = model(midpoints)
pred

tensor([[11.4022]], dtype=torch.float64)

#### Parity with BioAge

In [15]:
reference_predictions = params["reference"]
matrix = torch.tensor(
    [[row[name] for name in model.features] for row in reference_predictions["rows"]], dtype=torch.float64
)
with torch.inference_mode():
    predicted = model(matrix).squeeze(-1)

expected = torch.tensor(reference_predictions["expected"], dtype=torch.float64)
print("max absolute difference:", (predicted - expected).abs().max().item())

max absolute difference: 7.460698725481052e-13


## Save torch model

In [16]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [17]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: extract_homeostaticdysregulation.R
Deleted folder: Rlib
Deleted file: homeostaticdysregulation.json
